# HDP ML Properties — Results Overview

Visual browser for all model results. Loads pre-computed result pkl files — no re-running needed.

**Environment:** `add_bonding`

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from shared.pipeline import CorrelationFilter  # needed for unpickling
from shared.plotting import (
    plot_parity, plot_residuals,
    plot_aggregated_importance, plot_pareto,
)

def load(path):
    try:
        with open(path, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        print(f'Not found: {path}')
        return None

print('Imports OK')

## 1. Summary Table

In [ ]:
result_files = {
    'RF (Bandgap)':           '../models/results/results_rf_bandgap.pkl',
    'GB (Bandgap)':           '../models/results/results_gb_bandgap.pkl',
    'GB (Bandgap filtered)':  '../models/results/results_gb_bandgap_filtered.pkl',
    'GB (Eform)':             '../models/results/results_gb_eform.pkl',
    'GB (Ehull)':             '../models/results/results_gb_ehull.pkl',
    'GB (Ehull NoDOS)':       '../models/results/results_gb_ehull_nodos.pkl',
    'MODNet (Ehull)':         '../models/results/results_modnet_ehull.pkl',
}

results = {k: load(v) for k, v in result_files.items()}
results = {k: v for k, v in results.items() if v is not None}

rows = []
for label, res in results.items():
    rows.append({
        'Model': label,
        'CV R²': f"{res['cv_mean_r2']:.4f}",
        'CV std': f"±{res['cv_std_r2']:.4f}",
        'Holdout R²': f"{res['holdout_r2']:.4f}",
        'Gap': f"{res['holdout_r2'] - res['cv_mean_r2']:+.4f}",
        'Holdout MAE': f"{res['holdout_mae']:.4f}",
    })

pd.DataFrame(rows).set_index('Model')

## 2. Parity Plots

In [ ]:
plot_configs = [
    ('GB (Eform)',  'Eform', 'eV/atom', '#d4883a'),
    ('GB (Ehull)',  'Ehull', 'eV/atom', '#2b6c8f'),
    ('GB (Bandgap)', 'Bandgap', 'eV',  '#2b6c8f'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (label, tname, unit, color) in zip(axes, plot_configs):
    if label not in results:
        ax.set_visible(False)
        continue
    res = results[label]
    plot_parity(
        res['holdout_y_true'], res['holdout_y_pred'],
        res['holdout_r2'],
        title=f'{tname} holdout',
        xlabel=f'True {tname} ({unit})',
        ylabel=f'Predicted {tname} ({unit})',
        ax=ax, color=color
    )

plt.tight_layout()
plt.show()

## 3. Feature Importance: Eform vs Ehull

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (label, color, title) in zip(axes, [
    ('GB (Eform)', '#d4883a', 'Eform — Top features'),
    ('GB (Ehull)', '#2b6c8f', 'Ehull — Top features'),
]):
    if label not in results:
        ax.set_visible(False)
        continue

    res = results[label]
    all_imp = {}
    for fr in res['fold_results']:
        if 'feature_importances' not in fr:
            continue
        for name, val in fr['feature_importances'].items():
            all_imp.setdefault(name, []).append(val)

    if not all_imp:
        ax.text(0.5, 0.5, 'No importance data', ha='center', transform=ax.transAxes)
        continue

    avg = {k: np.mean(v) for k, v in all_imp.items()}
    top = sorted(avg.items(), key=lambda x: x[1], reverse=True)[:15]
    names = [n.split('|')[-1][:35] for n, _ in top]
    vals = [v for _, v in top]

    y = np.arange(len(names))
    ax.barh(y, vals, color=color, alpha=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(names, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Mean importance')
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 4. PySR Pareto Fronts

In [ ]:
import os

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pareto_configs = [
    ('ehull', 0.009542, axes[0], 'Ehull: PySR Pareto Fronts'),
    ('eform', 0.454758, axes[1], 'Eform: PySR Pareto Fronts'),
]

colors = ['#2b6c8f', '#d4883a', '#2e7d32']
modes = ['normal', 'corrfilter', 'focused']
labels = ['Normal', 'CorrFilter', 'Focused']

for target, y_var, ax, title in pareto_configs:
    for mode, label, color in zip(modes, labels, colors):
        path = f'../symbolic_regression/equations/pysr_{target}_{mode}_equations.csv'
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path).sort_values('complexity')
        r2s = 1 - df['loss'] / y_var
        ax.plot(df['complexity'], r2s, 'o-', label=label,
                color=color, markersize=3)

    ax.set_xlabel('Complexity')
    ax.set_ylabel('R²')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Per-Fold CV Performance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

plot_models = [
    ('GB (Bandgap)',  '#2b6c8f', 'o-'),
    ('GB (Eform)',    '#d4883a', 's-'),
    ('GB (Ehull)',    '#2b6c8f', '^--'),
    ('MODNet (Ehull)', '#7b1fa2', 'D--'),
]

for label, color, style in plot_models:
    if label not in results:
        continue
    res = results[label]
    folds = [fr['fold'] for fr in res['fold_results']]
    r2s   = [fr['r2']   for fr in res['fold_results']]
    ax.plot(folds, r2s, style, color=color, label=label, markersize=6)

ax.set_xlabel('Fold')
ax.set_ylabel('R²')
ax.set_title('Per-fold R² across models')
ax.legend()
ax.set_xticks([1, 2, 3, 4, 5])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Holdout vs CV Gap

In [ ]:
labels_plot, gaps, colors_gap = [], [], []

for label, res in results.items():
    gap = res['holdout_r2'] - res['cv_mean_r2']
    labels_plot.append(label)
    gaps.append(gap)
    colors_gap.append('#2e7d32' if gap >= 0 else '#c62828')

fig, ax = plt.subplots(figsize=(10, 4))
y = np.arange(len(labels_plot))
ax.barh(y, gaps, color=colors_gap, alpha=0.8)
ax.axvline(0, color='k', linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(labels_plot)
ax.set_xlabel('Holdout R² − CV Mean R²')
ax.set_title('CV–Holdout gap (green = CV conservative, red = CV optimistic)')
plt.tight_layout()
plt.show()